# hmslib — detectability and POD curves

How strong must a fault be before the detector sees it?

The intensity is swept inside the Monte Carlo loop, so at a given intensity the
runs still scatter: detection is a random event, and the answer is a
**probability of detection** as a function of intensity. This notebook produces,
per failure class:

* `i50` — intensity detected half of the time;
* `i90` — intensity detected 90% of the time;
* `i90_95` — upper 95% confidence bound on `i90`, the conservative number.

**A POD curve only means something at a stated false positive rate.** Lowering
the threshold buys detection for free, so every comparison here is made at one
fixed FPR, set through `AT_ALPHA`.

Prerequisite: an intensity column. If `op.intensity` is `None`, set
`columns.intensity` in the manifest first.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import hmslib as hm

hm.apply_style()
hm.set_seed(0)

AT_ALPHA = 1e-3      # the false positive rate every result below holds at

In [ ]:
MANIFEST = "manifest.json"        # produced by notebook 01

if not os.path.exists(MANIFEST):
    demo = os.path.join(os.path.expanduser("~"), "hmslib_demo_data")
    hm.synth.write_dataset(demo, n_ops=2, n_sensors=14, n_nominal=3000,
                           n_classes=6, n_per_class=600, effect_scale=0.05,
                           intensity_max=60.0, random_state=0)
    hm.io.scan_folder(demo, write=MANIFEST)

ds = hm.Dataset.from_manifest(MANIFEST, verbose=False)
op = ds[ds.operating_points[0]]
print(op.summary())
assert op.intensity is not None, "set columns.intensity in the manifest first"

## 1. Detector, calibrated at a stated FPR

The threshold comes from the nominal cloud alone. Everything downstream is
conditional on this number.

In [ ]:
bank = hm.ModelBank.fit(ds, hm.Mahalanobis(threshold="empirical", alpha=AT_ALPHA))
det = bank[op.name]
print("threshold %.4g  ->  FPR on nominal %.3f%%"
      % (det.threshold_, 100 * det.false_positive_rate(op.nominal[det.features_used_])))

## 2. Score against intensity

The raw picture. Where the median line of a class crosses the threshold is,
visually, its detectability limit; the spread around it is why that limit is a
probability rather than a number.

In [ ]:
fig = hm.viz.plot_score_vs_intensity(det, op, classes=op.classes[:6], at_alpha=AT_ALPHA)
plt.show()

## 3. POD curves

Points are the empirical detection rate over quantile bins of intensity, with
Wilson intervals; the line is the fitted log-odds model
`logit(POD) = b0 + b1·log(intensity)`. Dotted vertical marks are `i90`, dashed
ones `i90_95`.

In [ ]:
results = hm.analysis.pod_analysis(det, op, at_alpha=AT_ALPHA, n_boot=200)
fig = hm.viz.plot_pod({k: results[k] for k in op.classes[:6]})
plt.show()

## 4. The headline table

Hardest classes first: these are the weak spots of the monitoring system, the
faults that must grow largest before being noticed.

Read `i90_emp` next to `i90`: the first is non parametric, the second assumes
the log-odds model. When they disagree, the model assumption is the suspect,
not the data.

In [ ]:
table = hm.analysis.pod_table(results)
table.round(3)

In [ ]:
# anything the analysis wants to warn about
for cls, res in results.items():
    for note in res.notes:
        print("[%s] %s" % (cls, note))

### Detectability is not free — check the trade

Recomputing at several false positive rates shows what a looser alarm buys.
This is the curve to bring to a design review when someone asks for a lower
detection limit.

In [ ]:
rows = []
for alpha in (1e-4, 1e-3, 1e-2, 5e-2):
    res_a = hm.analysis.pod_analysis(det, op, at_alpha=alpha, n_boot=0)
    i90 = [r.i90 for r in res_a.values() if np.isfinite(r.i90)]
    rows.append({
        "alpha": alpha,
        "median i90": np.median(i90) if i90 else np.nan,
        "worst i90": np.max(i90) if i90 else np.nan,
        "classes with a finite i90": len(i90),
    })
pd.DataFrame(rows).round(3)

## 5. Which sensor reacts first

`slope` is sigmas gained per unit of intensity, `r` how cleanly the sensor
tracks the fault rather than the Monte Carlo scatter. This is the input for any
later discussion about sensor selection.

In [ ]:
sens = hm.analysis.sensor_sensitivity(op, classes=op.classes[:6])
fig = hm.viz.plot_sensor_sensitivity(sens, metric="slope", top=8)
plt.show()

sens.groupby("class").head(3).round(3)

## 6. Isolation: who carries the distance

The squared distance decomposes exactly over sensors. Averaged over the flagged
runs of a class, it names the culprits — the bridge from detection to diagnosis.

In [ ]:
fig = hm.viz.plot_contribution_heatmap(det, op, classes=op.classes[:10])
plt.show()

## 7. The report

One PDF per operating point with all of the above, plus the detector
diagnostics and the warnings.

In [ ]:
hm.detection_report(bank, ds, out="reports/detection.pdf", at_alpha=AT_ALPHA)

## 8. Where the open-set stage will start

Runs whose intensity is well below `i90` are, by construction, indistinguishable
from nominal. Training a diagnosis stage on them would make its metrics measure
the overlap with nominal rather than the ability to tell faults apart, so the
second stage is trained only on what the first stage promotes.

The table below is the intensity floor per class that F4 will use.

In [ ]:
floor = table[["class", "i50", "i90", "i90_95"]].copy()
floor["train above"] = floor[["i50", "i90"]].min(axis=1)
floor.round(3)